# Clase: Correlación, Asociación y Multicolinealidad

**Objetivo de la sesión**  
Entender, calcular e interpretar los coeficientes de Pearson, Spearman y Kendall, y detectar multicolinealidad mediante el VIF.

Este notebook es **guiado**: lo desarrollamos juntos en clase. Los alumnos tienen un notebook paralelo con ejercicios para practicar.

## 1. Preparación del entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)

## 2. Generación de datos sintéticos controlados

Creamos tres escenarios para ilustrar diferencias entre los coeficientes:

- **Escenario A**: relación lineal fuerte
- **Escenario B**: relación monotónica no lineal
- **Escenario C**: relación no monotónica (forma de U)

In [ ]:
n = 200

# Escenario A: lineal
x_a = np.random.normal(50, 15, n)
y_a = 2.5 * x_a + np.random.normal(0, 20, n)

# Escenario B: monotónica no lineal (exponencial)
x_b = np.random.uniform(1, 10, n)
y_b = np.exp(0.4 * x_b) + np.random.normal(0, 5, n)

# Escenario C: no monotónica (parábola)
x_c = np.random.uniform(-5, 5, n)
y_c = x_c ** 2 + np.random.normal(0, 3, n)

df_a = pd.DataFrame({"x": x_a, "y": y_a})
df_b = pd.DataFrame({"x": x_b, "y": y_b})
df_c = pd.DataFrame({"x": x_c, "y": y_c})

### Visualización rápida de los tres escenarios

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(df_a["x"], df_a["y"], alpha=0.6, edgecolor="none")
axes[0].set_title("A · Lineal")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")

axes[1].scatter(df_b["x"], df_b["y"], alpha=0.6, edgecolor="none", color="C1")
axes[1].set_title("B · Monotónica no lineal")
axes[1].set_xlabel("x"); axes[1].set_ylabel("y")

axes[2].scatter(df_c["x"], df_c["y"], alpha=0.6, edgecolor="none", color="C2")
axes[2].set_title("C · No monotónica (U)")
axes[2].set_xlabel("x"); axes[2].set_ylabel("y")

plt.tight_layout()
plt.show()

## 3. Cálculo de los tres coeficientes

Usamos `scipy.stats`:
- `pearsonr`
- `spearmanr`
- `kendalltau`

In [ ]:
def correlaciones(x, y, nombre=""):
    r_p, p_p = stats.pearsonr(x, y)
    r_s, p_s = stats.spearmanr(x, y)
    r_k, p_k = stats.kendalltau(x, y)
    print(f"=== {nombre} ===")
    print(f"Pearson  r  = {r_p:7.3f}   (p = {p_p:.3e})")
    print(f"Spearman ρ  = {r_s:7.3f}   (p = {p_s:.3e})")
    print(f"Kendall  τ  = {r_k:7.3f}   (p = {p_k:.3e})")
    print()

correlaciones(df_a["x"], df_a["y"], "Escenario A – Lineal")
correlaciones(df_b["x"], df_b["y"], "Escenario B – Monotónica no lineal")
correlaciones(df_c["x"], df_c["y"], "Escenario C – No monotónica")

**Discusión en clase**

- ¿Qué pasa en el escenario B con Pearson vs Spearman/Kendall?
- ¿Qué ocurre en el escenario C? ¿Alguno de los tres coeficientes captura bien la relación?
- ¿Por qué Kendall suele dar valores numéricamente más bajos que Spearman?

## 4. Matriz de correlación y heatmap

En la práctica trabajamos con muchas variables. Construimos un dataset con varias columnas y visualizamos la matriz.

In [ ]:
# Dataset con varias variables (algunas correlacionadas)
n = 300
ingreso = np.random.normal(4000, 1200, n)
gasto = 0.65 * ingreso + np.random.normal(0, 400, n)
ahorro = ingreso - gasto + np.random.normal(0, 200, n)
edad = np.random.randint(22, 65, n)
score_credito = 0.3 * ingreso / 100 + 0.4 * edad + np.random.normal(0, 15, n)

df = pd.DataFrame({
    "ingreso": ingreso,
    "gasto": gasto,
    "ahorro": ahorro,
    "edad": edad,
    "score_credito": score_credito
})

df.head()

In [ ]:
# Matriz de correlación de Pearson
corr_pearson = df.corr(method="pearson")
corr_spearman = df.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=axes[0], square=True)
axes[0].set_title("Pearson")

sns.heatmap(corr_spearman, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=axes[1], square=True)
axes[1].set_title("Spearman")

plt.tight_layout()
plt.show()

## 5. Multicolinealidad y VIF

El **Factor de Inflación de la Varianza (VIF)** cuantifica cuánto se infla la varianza de un coeficiente por la correlación con las demás predictoras.

$$\text{VIF}_i = \frac{1}{1 - R_i^2}$$

donde $R_i^2$ proviene de regresar la variable $i$ sobre el resto de predictoras.

In [ ]:
def calcular_vif(dataframe):
    """Calcula VIF para cada columna numérica."""
    X = add_constant(dataframe)
    vif_data = pd.DataFrame({
        "variable": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })
    return vif_data.query("variable != 'const'").sort_values("VIF", ascending=False)

print("VIF del dataset completo:")
print(calcular_vif(df).round(2))

**Interpretación de umbrales comunes**

| VIF        | Interpretación                  |
|------------|---------------------------------|
| ≈ 1        | Sin multicolinealidad relevante |
| 1 – 5      | Aceptable en la mayoría de casos|
| 5 – 10     | Precaución                      |
| > 10       | Multicolinealidad problemática  |

### Experimento: forzar multicolinealidad alta

Creamos una variable casi colineal y observamos el cambio en los VIF.

In [ ]:
df_col = df.copy()
df_col["gasto_casi_igual"] = df_col["gasto"] * 0.98 + np.random.normal(0, 30, len(df_col))

print("VIF con variable casi colineal añadida:")
print(calcular_vif(df_col).round(2))

## 6. Resumen de la clase

1. **Pearson** → relación *lineal*. Supuestos más fuertes.
2. **Spearman / Kendall** → relación *monotónica*. Más robustos.
3. Ningún coeficiente captura bien relaciones no monotónicas complejas.
4. **Correlación ≠ causalidad**.
5. **VIF** es la herramienta estándar para detectar multicolinealidad entre predictoras.
6. Multicolinealidad afecta principalmente la *interpretabilidad* e *inferencia*, no necesariamente el poder predictivo global.

---

**Siguiente actividad**: abrir el notebook de ejercicios de los alumnos y resolver en grupos.